# tox21 practice

## Icebreak Rdkit

In [2]:
import pandas as pd

url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz"
df = pd.read_csv(url)

print(df.shape)
df.head()

(7831, 14)


,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O


In [5]:
import deepchem as dc

tasks, datasets, transformers = dc.molnet.load_tox21(featurizer='ECFP', splitter='random')
train_dataset, valid_dataset, test_dataset = datasets

print("Tasks (12개 assay):", tasks)
print("Train 크기:", len(train_dataset))
print("Valid 크기:", len(valid_dataset))
print("Test 크기:", len(test_dataset))

ModuleNotFoundError: No module named 'deepchem'

In [6]:
# 에러 나와서 새로 파싱하는 과정 필요
from rdkit import Chem

def is_valid_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return mol is not None

df['valid'] = df['smiles'].apply(is_valid_smiles)

n_total = len(df)
n_valid = df['valid'].sum()
n_invalid = n_total - n_valid

print(f"전체: {n_total}개, 파싱 성공: {n_valid}개, 파싱 실패(제외): {n_invalid}개")

# 나중에 투명성 기록용으로 실패한 분자 목록도 따로 저장
invalid_smiles = df[~df['valid']]['smiles'].tolist()

df_clean = df[df['valid']].reset_index(drop=True)

ModuleNotFoundError: No module named 'rdkit'

In [ ]:
from rdkit.Chem import AllChem
import numpy as np

def smiles_to_ecfp(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp)

df_clean['fingerprint'] = df_clean['smiles'].apply(smiles_to_ecfp)
print(df_clean['fingerprint'].iloc[0].shape)  # (2048,) 이 나와야 정상

In [ ]:
# 12개 assay 컬럼명 확인 (smiles, mol_id 등을 제외한 나머지)
task_cols = [c for c in df_clean.columns if c not in ['smiles', 'mol_id', 'valid', 'fingerprint']]
print("Tasks:", task_cols)
print("개수:", len(task_cols))  # 12가 나와야 정상

import numpy as np

# y: NaN을 일단 0으로 채움 (단, w로 가려질 것이므로 실제 학습엔 영향 없음)
y = df_clean[task_cols].fillna(0).values.astype(np.float32)

# w: NaN이 아니면 1(유효), NaN이면 0(무효)
w = (~df_clean[task_cols].isna()).values.astype(np.float32)

print("y shape:", y.shape)  # (7823, 12) 나와야 함
print("w shape:", w.shape)  # (7823, 12) 나와야 함

# 각 assay별로 실제 측정된 비율 확인
for i, task in enumerate(task_cols):
    coverage = w[:, i].mean()
    positive_rate = y[w[:, i] == 1, i].mean()  # 측정된 것들 중 양성 비율
    print(f"{task}: 측정률 {coverage:.1%}, 그 중 양성 비율 {positive_rate:.1%}")

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

X = np.stack(df_clean['fingerprint'].values)  # (7823, 2048)

indices = np.arange(len(X))
train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42)
valid_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

X_train, y_train, w_train = X[train_idx], y[train_idx], w[train_idx]
X_valid, y_valid, w_valid = X[valid_idx], y[valid_idx], w[valid_idx]
X_test, y_test, w_test = X[test_idx], y[test_idx], w[test_idx]

print("Train:", X_train.shape, "Valid:", X_valid.shape, "Test:", X_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

auc_scores = {}

for i, task in enumerate(task_cols):
    # w=1인(측정된) 샘플만 사용
    train_mask = w_train[:, i] == 1
    valid_mask = w_valid[:, i] == 1

    clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    clf.fit(X_train[train_mask], y_train[train_mask, i])

    probs = clf.predict_proba(X_valid[valid_mask])[:, 1]
    auc = roc_auc_score(y_valid[valid_mask, i], probs)
    auc_scores[task] = auc
    print(f"{task}: AUROC = {auc:.3f}")

print(f"\n평균 AUROC: {np.mean(list(auc_scores.values())):.3f}")

In [ ]:
from google.colab import userdata

# Colab Secrets에서 GitHub Token 가져오기
token = userdata.get("GITHUB_TOKEN")

# Repository 정보
owner = "Dec32th"
repo = "laidd-2026"

# Private Repository Clone
!git clone https://{token}@github.com/{owner}/{repo}.git

# Repository로 이동
%cd {repo}

In [ ]:
!ls